In [43]:
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset

from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from transformers import AutoModel, AutoTokenizer

In [ ]:
data = load_dataset('cornell-movie-review-data/rotten_tomatoes')

In [3]:
print(f"Training split : {len(data['train'])}")
print(f"Testing split : {len(data['test'])}")
print(f"validation split : {len(data['validation'])}")

Training split : 8530
Testing split : 1066
validation split : 1066


In [4]:
for i in map(int, np.random.randint(0, len(data['train']), 2)):
    print(f"Text {i}: {data['train']['text'][i]}")
    print(f"Label {i}: {data['train']['label'][i]}\n")

Text 3744: although made on a shoestring and unevenly acted , conjures a lynch-like vision of the rotting underbelly of middle america .
Label 3744: 1

Text 1391: it is the sheer , selfish , wound-licking , bar-scrapping doggedness of leon's struggle to face and transmute his demons that makes the movie a spirited and touching occasion , despite its patchy construction .
Label 1391: 1



In [ ]:
model_name ="sentence-transformers/all-mpnet-base-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, device_map = 'auto')

In [25]:
def get_embeddings(texts):
    embeddings = []

    for i, text in enumerate(tqdm(texts, total=len(texts))):
      try:
        inputs = tokenizer(text, return_tensors="pt")

        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
        embeddings.append(embedding.cpu())

      except Exception as e:
        print(f"Error at index {i}: {e}")
        break

    return torch.stack(embeddings)

In [37]:
train_embeddings = get_embeddings(data["train"]["text"])
test_embeddings = get_embeddings(data["test"]["text"])

100%|██████████| 1066/1066 [00:12<00:00, 82.54it/s]


In [38]:
print(f"shape of Train Embeddings : {train_embeddings.shape}")
print(f"shape of Test Embeddings : {test_embeddings.shape}")

shape of Train Embeddings : torch.Size([8530, 768])
shape of Test Embeddings : torch.Size([1066, 768])


In [39]:
clf = LogisticRegression()
clf.fit(train_embeddings, data['train']['label'])

y_pred = clf.predict(test_embeddings)

In [40]:
def evaluate(y_true, y_pred):
  performance = classification_report(y_true, y_pred, target_names=['Negative Review', 'Positive Review'])
  print(performance)

In [41]:
evaluate(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.84      0.85      0.84       533
Positive Review       0.85      0.83      0.84       533

       accuracy                           0.84      1066
      macro avg       0.84      0.84      0.84      1066
   weighted avg       0.84      0.84      0.84      1066



In [42]:
labels = ["A negative review", "A positive review"]

inputs = tokenizer(labels, return_tensors="pt")

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

label_embeddings = outputs.last_hidden_state.mean(dim=1)

print(label_embeddings.shape)

torch.Size([2, 768])


In [45]:
similarity_matrix = cosine_similarity(test_embeddings.cpu().numpy(),
                                      label_embeddings.cpu().numpy())

y_pred = np.argmax(similarity_matrix, axis=1)

In [46]:
evaluate(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.78      0.77      0.78       533
Positive Review       0.77      0.79      0.78       533

       accuracy                           0.78      1066
      macro avg       0.78      0.78      0.78      1066
   weighted avg       0.78      0.78      0.78      1066

